# 🚗 Car Price Prediction — Multiple Linear Regression (CRISP-DM)

**Course:** AIoT Machine Learning HW3  
**Dataset:** [Car Price Prediction — Kaggle](https://www.kaggle.com/datasets/hellbuoy/car-price-prediction)  
**Algorithm:** Multiple Linear Regression  
**Target Variable:** `price` (USD)  

---

## CRISP-DM Six Phases

| # | Phase | Description |
|---|-------|-------------|
| 1 | Business Understanding | Define business value of price prediction |
| 2 | Data Understanding | EDA + missing value analysis |
| 3 | Data Preparation | OHE + Standardization + RFE Feature Selection |
| 4 | Modeling | statsmodels OLS + sklearn LinearRegression |
| 5 | Evaluation | RMSE / R² + Confidence Interval visualization |
| 6 | Deployment | API architecture + future improvements |

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Install packages if needed (uncomment to run)
# ─────────────────────────────────────────────────────────────
# !pip install pandas numpy matplotlib seaborn scikit-learn statsmodels scipy joblib

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import joblib

# ── Plot settings ─────────────────────────────────────────────
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('✅ All packages loaded successfully')

---
## Phase 1 — Business Understanding

### 📌 Background

Car pricing is a complex, multi-dimensional decision process involving:
- **Engineering specs**: engine displacement, horsepower, compression ratio, fuel type
- **Body design**: body style, wheelbase, car width, curb weight
- **Brand premium**: brand image, market positioning (luxury vs. economy)
- **Fuel efficiency**: city/highway mpg (related to environmental regulations)

### 🎯 Business Value

| Use Case | Business Value |
|---------|----------------|
| **Manufacturer Pricing** | Generate competitive MSRP from engineering specs, shorten pricing cycles |
| **Used-Car Platforms** | Provide fair market value (e.g., CarMax, Carvana), increase user trust |
| **Bank/Insurance Underwriting** | Objective vehicle valuation, reduce underwriting risk |

### 📊 Data Mining Goal

Build an interpretable **Multiple Linear Regression** model with:
- **R² ≥ 0.85** (explain at least 85% of price variance)
- **Final feature count strictly between 10 and 20**
- **All features with p-value < 0.05** (statistically significant)

---
## Phase 2 — Data Understanding

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.1  Load Dataset
# ─────────────────────────────────────────────────────────────
DATA_URL = (
    "https://raw.githubusercontent.com/dsrscientist/"
    "dataset1/master/CarPrice_Assignment.csv"
)

try:
    df = pd.read_csv("CarPrice_Assignment.csv")
    print("✅ Loaded from local file")
except FileNotFoundError:
    print("⚠ Local file not found, loading from URL...")
    df = pd.read_csv(DATA_URL)
    print("✅ Loaded from URL")

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.2  Data Info & Summary Statistics
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("Data Types & Non-Null Counts")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("Numeric Feature Summary Statistics")
print("=" * 60)
df.describe().round(2)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.3  Missing Value Check
# ─────────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df_filtered = missing_df[missing_df['Missing Count'] > 0]

if missing_df_filtered.empty:
    print("✅ No missing values found in this dataset!")
else:
    print("⚠ Missing values detected:")
    display(missing_df_filtered)

# Categorical column unique values
print("\n" + "=" * 60)
print("Categorical Columns — Unique Values")
print("=" * 60)
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    vals = df[col].unique()
    print(f"  {col:20s} ({len(vals):3d}): {list(vals[:6])}{'...' if len(vals) > 6 else ''}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.4  EDA — Target Variable Distribution
# ─────────────────────────────────────────────────────────────
print("price summary:")
print(df['price'].describe().round(2))
print(f"\nSkewness : {df['price'].skew():.4f}")
print(f"Kurtosis : {df['price'].kurt():.4f}")
print("\n→ price is right-skewed (skewness > 0); high-price vehicles pull the right tail")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Phase 2 -- Price Distribution (Target Variable)",
             fontsize=14, fontweight='bold')

axes[0].hist(df['price'], bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean: {df['price'].mean():,.0f}")
axes[0].axvline(df['price'].median(), color='orange', linestyle='--', linewidth=2,
                label=f"Median: {df['price'].median():,.0f}")
axes[0].set_title("Price Histogram (Original)", fontsize=12)
axes[0].set_xlabel("Price (USD)")
axes[0].set_ylabel("Frequency")
axes[0].legend(fontsize=9)

log_price = np.log1p(df['price'])
axes[1].hist(log_price, bins=30, color='#DD8452', edgecolor='white', alpha=0.85)
axes[1].set_title("log(price+1) Histogram (Skewness Correction)", fontsize=12)
axes[1].set_xlabel("log(Price + 1)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("eda_price_distribution.png", bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.5  EDA — Numeric Feature Correlation
# ─────────────────────────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_cols].corr()
price_corr = corr_matrix['price'].drop('price').sort_values(ascending=False)

print("Pearson Correlation with price (descending):")
print(price_corr.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Phase 2 -- Feature Correlation Analysis", fontsize=14, fontweight='bold')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[0], cmap='coolwarm',
            vmin=-1, vmax=1, center=0, annot=False,
            linewidths=0.3, cbar_kws={'shrink': 0.8})
axes[0].set_title("Numeric Feature Correlation Heatmap", fontsize=12)
axes[0].tick_params(axis='x', rotation=45, labelsize=7)
axes[0].tick_params(axis='y', rotation=0, labelsize=7)

colors = ['#c0392b' if v > 0 else '#2980b9' for v in price_corr.values]
axes[1].barh(range(len(price_corr)), price_corr.values, color=colors, alpha=0.82)
axes[1].set_yticks(range(len(price_corr)))
axes[1].set_yticklabels(price_corr.index, fontsize=9)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title("Pearson Correlation with Price", fontsize=12)
axes[1].set_xlabel("Pearson r")

plt.tight_layout()
plt.savefig("eda_correlation.png", bbox_inches='tight')
plt.show()

print(f"\nTop 5 features correlated with price:")
for feat, corr_val in price_corr.head(5).items():
    print(f"   {feat:20s}: r = {corr_val:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.6  EDA — Categorical Features vs Price (Box Plot)
# ─────────────────────────────────────────────────────────────
cat_to_plot = ['fueltype', 'aspiration', 'carbody', 'drivewheel', 'enginetype', 'cylindernumber']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Phase 2 -- Categorical Features vs Price (Box Plot)",
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(cat_to_plot):
    order = df.groupby(col)['price'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='price', order=order, ax=axes[i],
                palette='Set2', linewidth=1.2)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel("")
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)

plt.tight_layout()
plt.savefig("eda_categorical_boxplots.png", bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.7  EDA — Key Numeric Features vs Price (Scatter)
# ─────────────────────────────────────────────────────────────
num_features_plot = ['enginesize', 'horsepower', 'curbweight', 'carwidth', 'wheelbase']
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle("Phase 2 -- Key Numeric Features vs Price (Scatter Plot)",
             fontsize=13, fontweight='bold')
colors_list = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for i, feat in enumerate(num_features_plot):
    axes[i].scatter(df[feat], df['price'], alpha=0.35, s=20,
                    color=colors_list[i], edgecolors='white', linewidths=0.2)
    z = np.polyfit(df[feat], df['price'], 1)
    p = np.poly1d(z)
    xline = np.linspace(df[feat].min(), df[feat].max(), 200)
    axes[i].plot(xline, p(xline), color='red', linewidth=1.5, linestyle='--')
    r = df[feat].corr(df['price'])
    axes[i].set_title(f"{feat}\nr={r:.3f}", fontsize=9, fontweight='bold')
    axes[i].set_xlabel(feat, fontsize=7)
    axes[i].set_ylabel("price" if i == 0 else "", fontsize=8)

plt.tight_layout()
plt.savefig("eda_scatter_plots.png", bbox_inches='tight')
plt.show()

---
## Phase 3 — Data Preparation

### Processing Pipeline
1. Extract car brand from `CarName`
2. Drop redundant columns (`car_ID`, `CarName`)
3. One-Hot Encoding for categorical variables
4. Train/Test split (80% / 20%)
5. Z-Score Standardization
6. **Feature Selection (RFE → p-value pruning) → strictly 10–20 features**
7. VIF multicollinearity check

In [ ]:
# ─────────────────────────────────────────────────────────────
#  3.1  Extract Brand from CarName
# ─────────────────────────────────────────────────────────────
df_clean = df.copy()
df_clean['brand'] = df_clean['CarName'].str.split().str[0].str.lower()

# Fix typos in the dataset
brand_fix = {
    'maxda':     'mazda',
    'porcshce':  'porsche',
    'toyouta':   'toyota',
    'vokswagen': 'volkswagen',
    'vw':        'volkswagen',
}
df_clean['brand'] = df_clean['brand'].replace(brand_fix)

print(f"Total brands: {df_clean['brand'].nunique()}")
brand_price_avg = df_clean.groupby('brand')['price'].mean().sort_values(ascending=False)
print("\nAverage price by brand:")
print(brand_price_avg.round(0).to_string())

# Visualize brand premium
fig, ax = plt.subplots(figsize=(12, 6))
colors_brand = plt.cm.RdYlGn_r(np.linspace(0, 1, len(brand_price_avg)))
ax.bar(brand_price_avg.index, brand_price_avg.values, color=colors_brand, alpha=0.85)
ax.set_title("Average Price by Brand (Brand Premium Visualization)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Brand", fontsize=11)
ax.set_ylabel("Average Price (USD)", fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=8)
plt.tight_layout()
plt.savefig("eda_brand_price.png", bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  3.2  Drop Redundant Columns & One-Hot Encoding
# ─────────────────────────────────────────────────────────────
drop_cols = ['car_ID', 'CarName']
df_clean.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

cat_encode = [
    'fueltype', 'aspiration', 'doornumber', 'carbody',
    'drivewheel', 'enginelocation', 'enginetype',
    'cylindernumber', 'fuelsystem', 'brand'
]
cat_encode = [c for c in cat_encode if c in df_clean.columns]

df_encoded = pd.get_dummies(df_clean, columns=cat_encode, drop_first=True)

# Convert bool columns to int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"\nShape before OHE : {df_clean.shape}")
print(f"Shape after  OHE : {df_encoded.shape}")
print(f"Total features (before selection): {df_encoded.shape[1] - 1}")
print("\n→ Feature count far exceeds 20 → Feature Selection required!")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  3.3  Split X/y, Train/Test Split, Standardization
# ─────────────────────────────────────────────────────────────
target = 'price'
feature_cols = [c for c in df_encoded.columns if c != target]
X = df_encoded[feature_cols]
y = df_encoded[target]

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train set : {X_train.shape[0]} samples")
print(f"Test  set : {X_test.shape[0]} samples")

# Standardization (fit on train only to prevent data leakage)
num_features_to_scale = X.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[num_features_to_scale] = scaler.fit_transform(X_train[num_features_to_scale])
X_test_scaled[num_features_to_scale]  = scaler.transform(X_test[num_features_to_scale])

print(f"\nStandardized {len(num_features_to_scale)} numeric columns")
print("✅ Standardization complete (fit on train only — no data leakage)")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  3.4  Feature Selection — Step A: RFE
# ─────────────────────────────────────────────────────────────
TARGET_FEATURES = 15  # Target: 15 features (within 10–20)

print(f"▶ RFE target feature count: {TARGET_FEATURES}")
print("-" * 60)

lr_rfe = LinearRegression()
rfe = RFE(estimator=lr_rfe, n_features_to_select=TARGET_FEATURES, step=1)
rfe.fit(X_train_scaled, y_train)

rfe_selected   = X_train_scaled.columns[rfe.support_].tolist()
rfe_eliminated = X_train_scaled.columns[~rfe.support_].tolist()

print(f"✅ RFE kept ({len(rfe_selected)} features):")
for f in rfe_selected:
    print(f"   ├─ {f}")

print(f"\n❌ RFE eliminated ({len(rfe_eliminated)} features):")
for f in rfe_eliminated[:15]:
    print(f"   ├─ {f}")
if len(rfe_eliminated) > 15:
    print(f"   └─ ...and {len(rfe_eliminated)-15} more")

X_train_rfe = X_train_scaled[rfe_selected]
X_test_rfe  = X_test_scaled[rfe_selected]

In [ ]:
# ─────────────────────────────────────────────────────────────
#  3.5  Feature Selection — Step B: p-value Pruning
# ─────────────────────────────────────────────────────────────
print("▶ p-value pruning (iteratively remove p > 0.05, keep ≥ 10 features)")
print("-" * 60)

X_sm = sm.add_constant(X_train_rfe)
eliminated_pvalue = []
iteration = 0

while True:
    model_ols_sel = sm.OLS(y_train, X_sm).fit()
    pvalues = model_ols_sel.pvalues.drop('const')
    max_pval = pvalues.max()

    if max_pval > 0.05:
        worst_feature = pvalues.idxmax()
        current_count = len(pvalues)
        if current_count - 1 < 10:
            print(f"\n⚠ Stopped: removing one more would drop below 10 features (currently {current_count})")
            break
        eliminated_pvalue.append(worst_feature)
        X_sm = X_sm.drop(columns=[worst_feature])
        iteration += 1
        print(f"  Iter {iteration:2d}: removed '{worst_feature}' (p={max_pval:.4f})")
    else:
        print(f"\n✅ All remaining features have p-value < 0.05 (after {iteration} iterations)")
        break

final_features = [c for c in X_sm.columns if c != 'const']
final_count    = len(final_features)

print(f"\n{'='*60}")
print(f"📋 Final feature count: {final_count} (within 10–20 requirement)")
print(f"{'='*60}")
print(f"\n✅ Final kept features:")
for i, f in enumerate(final_features):
    print(f"   {i+1:2d}. {f}")
print(f"\n❌ Additionally removed by p-value ({len(eliminated_pvalue)}): {eliminated_pvalue}")

assert 10 <= final_count <= 20, f"❌ Feature count {final_count} not in 10–20 range!"
print(f"\n✅ Assertion passed: {final_count} features (satisfies 10–20 constraint)")

X_train_final = X_train_scaled[final_features]
X_test_final  = X_test_scaled[final_features]

In [ ]:
# ─────────────────────────────────────────────────────────────
#  3.6  VIF Multicollinearity Check
# ─────────────────────────────────────────────────────────────
X_vif = sm.add_constant(X_train_final)
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF':     [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif_data = vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False)

print("VIF Multicollinearity Results:")
print(vif_data.to_string(index=False))
print("\nThreshold: VIF < 5 (low)  |  5 ≤ VIF < 10 (moderate)  |  VIF ≥ 10 (severe)")

fig, ax = plt.subplots(figsize=(10, 6))
colors_vif = ['#c0392b' if v > 10 else '#e67e22' if v > 5 else '#27ae60'
              for v in vif_data['VIF']]
ax.barh(vif_data['Feature'], vif_data['VIF'], color=colors_vif, alpha=0.85)
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='VIF=5 (Moderate)')
ax.axvline(10, color='red',    linestyle='--', linewidth=1.5, label='VIF=10 (Severe)')
ax.set_title("Phase 3 -- VIF Multicollinearity Check (Final Features)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("VIF Score", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("feature_selection_vif.png", bbox_inches='tight')
plt.show()

---
## Phase 4 — Modeling

| Framework | Purpose |
|-----------|----------|
| `statsmodels OLS` | Full statistical report (R², F-test, p-value, AIC, Durbin-Watson) |
| `sklearn LinearRegression` | Prediction, evaluation, model persistence |

In [ ]:
# ─────────────────────────────────────────────────────────────
#  4.1  statsmodels OLS — Full Statistical Report
# ─────────────────────────────────────────────────────────────
X_train_sm = sm.add_constant(X_train_final)
X_test_sm  = sm.add_constant(X_test_final)

ols_model = sm.OLS(y_train, X_train_sm).fit()

print("=" * 70)
print("statsmodels OLS Full Statistical Report")
print("=" * 70)
print(ols_model.summary())
print("\n💡 Key Interpretations:")
print("  - R-squared / Adj. R-squared : Overall model explanatory power")
print("  - F-statistic p-value        : Overall model significance (should be < 0.05)")
print("  - P>|t| per feature          : Individual feature significance (< 0.05 = significant)")
print("  - Durbin-Watson              : Residual autocorrelation (≈2 means none)")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  4.2  sklearn LinearRegression — Coefficient Visualization
# ─────────────────────────────────────────────────────────────
sk_model = LinearRegression()
sk_model.fit(X_train_final, y_train)

coef_df = pd.DataFrame({
    'Feature':     final_features,
    'Coefficient': sk_model.coef_
}).sort_values('Coefficient', ascending=True)

print(f"Intercept: {sk_model.intercept_:,.2f}")
print("\nRegression coefficients (sorted by absolute value):")
print(coef_df.sort_values('Coefficient', key=abs, ascending=False).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 7))
colors_coef = ['#c0392b' if v > 0 else '#2980b9' for v in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'],
        color=colors_coef, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_title("Phase 4 -- Regression Coefficients (Standardized Features)\n"
             "Red = Positive effect (price up)  |  Blue = Negative effect (price down)",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Coefficient Value (Standardized)", fontsize=11)

for bar, val in zip(ax.patches, coef_df['Coefficient']):
    ax.text(val + (50 if val >= 0 else -50), bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig("model_coefficients.png", bbox_inches='tight')
plt.show()
print("\n💡 Larger absolute coefficient = stronger price influence")

---
## Phase 5 — Evaluation

| Metric | Description |
|--------|-------------|
| **R²** | Proportion of variance explained (1 = perfect) |
| **Adjusted R²** | R² corrected for number of features |
| **RMSE** | Root Mean Squared Error (same unit as price, USD) |
| **MAE** | Mean Absolute Error (less sensitive to outliers) |
| **MAPE** | Mean Absolute Percentage Error |

In [ ]:
# ─────────────────────────────────────────────────────────────
#  5.1  Compute Evaluation Metrics
# ─────────────────────────────────────────────────────────────
y_train_pred = sk_model.predict(X_train_final)
y_test_pred  = sk_model.predict(X_test_final)

def evaluate(y_true, y_pred, n_features, label=""):
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    r2     = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / (len(y_true) - n_features - 1)
    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  R²            = {r2:.4f}")
    print(f"  Adjusted R²   = {adj_r2:.4f}")
    print(f"  RMSE          = {rmse:>10,.2f} USD")
    print(f"  MAE           = {mae:>10,.2f} USD")
    print(f"  MAPE          = {mape:>9.2f}%")
    return {'R2': r2, 'Adj_R2': adj_r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

train_metrics = evaluate(y_train, y_train_pred, final_count, "Train Set")
test_metrics  = evaluate(y_test,  y_test_pred,  final_count, "Test Set")

print(f"\n{'='*50}")
print(f"  Overfitting Diagnosis")
print(f"{'='*50}")
gap = abs(train_metrics['R2'] - test_metrics['R2'])
print(f"  Train R² - Test R² = {gap:.4f}")
if gap < 0.05:
    print("  ✅ Gap < 0.05: No significant overfitting")
elif gap < 0.10:
    print("  ⚠ Gap 0.05–0.10: Mild overfitting")
else:
    print("  ❌ Gap > 0.10: Significant overfitting detected")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  5.2  ⭐ Actual vs Predicted — with 95% Confidence Interval
#       Achieved via seaborn.regplot(ci=95)
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

# seaborn.regplot: scatter + regression line + 95% CI (light blue shading)
sns.regplot(
    x=y_test.values,
    y=y_test_pred,
    ax=ax,
    scatter_kws={
        'alpha': 0.55, 's': 65,
        'color': '#4C72B0',
        'edgecolors': 'white', 'linewidths': 0.5, 'zorder': 3,
    },
    line_kws={
        'color': '#c0392b', 'linewidth': 2.5,
        'label': 'Regression Line', 'zorder': 4,
    },
    ci=95,          # ← 95% Confidence Interval (light blue shading)
    color='#c0392b'
)

# Perfect prediction reference line (y = x)
lim_min = min(y_test.min(), y_test_pred.min()) * 0.85
lim_max = max(y_test.max(), y_test_pred.max()) * 1.10
ax.plot([lim_min, lim_max], [lim_min, lim_max],
        'k--', linewidth=1.8, alpha=0.65, label='Perfect Prediction (y=x)', zorder=2)

ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.set_title(
    f"Phase 5 -- Actual vs Predicted Price (Test Set, n={len(y_test)})\n"
    f"R² = {test_metrics['R2']:.4f}  |  RMSE = ${test_metrics['RMSE']:,.0f}  |  "
    f"with 95% Confidence Interval",
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel("Actual Price (USD)", fontsize=12)
ax.set_ylabel("Predicted Price (USD)", fontsize=12)

ax.text(
    0.03, 0.97,
    "🔵 Scatter = test samples\n"
    "─── Red line = regression trend\n"
    "░░░ Shaded area = 95% Confidence Interval\n"
    "--- Black dashed = perfect prediction (y=x)",
    transform=ax.transAxes, fontsize=9,
    va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow',
              edgecolor='gray', alpha=0.9)
)

ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig("eval_actual_vs_predicted.png", bbox_inches='tight', dpi=150)
plt.show()
print("\n💡 95% CI interpretation:")
print("  If we repeat the sampling & fitting 100 times, the regression line")
print("  would fall within the shaded band 95% of the time.")
print("  Narrower band → more precise regression estimate.")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  5.3  Residual Diagnostic Plots (2×2 Grid)
# ─────────────────────────────────────────────────────────────
residuals_train = y_train - y_train_pred
residuals_test  = y_test  - y_test_pred

print("Residual Statistics (Train Set):")
print(f"  Mean   : {residuals_train.mean():.4f} (should be ≈ 0)")
print(f"  Skew   : {stats.skew(residuals_train):.4f}")
stat, pv = stats.shapiro(residuals_train.values[:50])
print(f"  Shapiro-Wilk p-value (first 50): {pv:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Phase 5 -- Residual Diagnostics (Test Set)",
             fontsize=14, fontweight='bold')

# Plot 1: Residuals vs Fitted
axes[0, 0].scatter(y_test_pred, residuals_test, alpha=0.5, color='#4C72B0', s=40,
                   edgecolors='white', linewidths=0.3)
axes[0, 0].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[0, 0].set_title("Residuals vs Fitted Values", fontsize=10, fontweight='bold')
axes[0, 0].set_xlabel("Predicted Price")
axes[0, 0].set_ylabel("Residuals")

# Plot 2: Residuals Histogram
axes[0, 1].hist(residuals_test, bins=20, color='#55A868', edgecolor='white', alpha=0.82)
axes[0, 1].set_title("Residuals Histogram", fontsize=10, fontweight='bold')
axes[0, 1].set_xlabel("Residuals")
axes[0, 1].set_ylabel("Frequency")

# Plot 3: Q-Q Plot
sm.qqplot(residuals_test, line='s', ax=axes[1, 0], alpha=0.55,
          markerfacecolor='#C44E52', markersize=5, markeredgewidth=0)
axes[1, 0].set_title("Q-Q Plot (Normality Check)", fontsize=10, fontweight='bold')

# Plot 4: Scale-Location
std_resid = np.sqrt(np.abs(residuals_test / residuals_test.std()))
axes[1, 1].scatter(y_test_pred, std_resid, alpha=0.5, color='#8172B2', s=40,
                   edgecolors='white', linewidths=0.3)
axes[1, 1].set_title("Scale-Location Plot", fontsize=10, fontweight='bold')
axes[1, 1].set_xlabel("Predicted Price")
axes[1, 1].set_ylabel("sqrt(|Standardized Residuals|)")

plt.tight_layout()
plt.savefig("eval_residual_diagnostics.png", bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  5.4  Metrics Comparison Bar Chart (Train vs Test)
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Phase 5 -- Model Performance Metrics Comparison",
             fontsize=13, fontweight='bold')

r2_vals    = [train_metrics['R2'],    test_metrics['R2']]
adjr2_vals = [train_metrics['Adj_R2'], test_metrics['Adj_R2']]
x = np.arange(2)
w = 0.35

b1 = axes[0].bar(x - w/2, r2_vals,    w, label='R²',      color='#4C72B0', alpha=0.85)
b2 = axes[0].bar(x + w/2, adjr2_vals, w, label='Adj. R²', color='#DD8452', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Train Set', 'Test Set'], fontsize=11)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("R-squared and Adjusted R-squared", fontsize=11)
axes[0].legend()
axes[0].axhline(0.85, color='red', linestyle='--', linewidth=1, label='Target R²=0.85')
for bar in list(b1) + list(b2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=9)

train_err = [train_metrics['RMSE'], train_metrics['MAE']]
test_err  = [test_metrics['RMSE'],  test_metrics['MAE']]
x2 = np.arange(2)

b3 = axes[1].bar(x2 - w/2, train_err, w, label='Train Set', color='#55A868', alpha=0.85)
b4 = axes[1].bar(x2 + w/2, test_err,  w, label='Test Set',  color='#C44E52', alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(['RMSE', 'MAE'], fontsize=11)
axes[1].set_title("RMSE and MAE (USD)", fontsize=11)
axes[1].legend()
for bar in list(b3) + list(b4):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{bar.get_height():,.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig("eval_metrics_comparison.png", bbox_inches='tight')
plt.show()

# Final summary
print(f"\n{'='*60}")
print(f"  📊 Final Model Evaluation Summary")
print(f"{'='*60}")
print(f"  Feature count      : {final_count} (satisfies 10–20 constraint)")
print(f"  R² (Test)          : {test_metrics['R2']:.4f} {'✅ Goal achieved' if test_metrics['R2'] >= 0.85 else '⚠ Below target'}")
print(f"  Adjusted R² (Test) : {test_metrics['Adj_R2']:.4f}")
print(f"  RMSE (Test)        : ${test_metrics['RMSE']:,.2f}")
print(f"  MAE  (Test)        : ${test_metrics['MAE']:,.2f}")
print(f"  MAPE (Test)        : {test_metrics['MAPE']:.2f}%")
print(f"  Overfitting gap    : {abs(train_metrics['R2'] - test_metrics['R2']):.4f}")

---
## Phase 6 — Deployment

### 🚀 Real-World Application Scenarios

#### 1. Real-Time Valuation API for Car Marketplaces
Wrap the trained model as a **RESTful API** (FastAPI recommended) and integrate with used-car platforms.

```python
# Deployment example (FastAPI)
from fastapi import FastAPI
import joblib, pandas as pd

app = FastAPI()
artifacts = joblib.load('car_price_model.pkl')
model  = artifacts['sk_model']
scaler = artifacts['scaler']
feats  = artifacts['final_features']

@app.post('/predict')
def predict(data: dict):
    df_input  = pd.DataFrame([data])
    df_scaled = scaler.transform(df_input[feats])
    price_pred = model.predict(df_scaled)[0]
    return {'predicted_price': round(price_pred, 2)}
```

#### 2. Deployment Architecture

```
User (Browser / App)
      │  POST /predict { enginesize, horsepower, brand, ... }
      ▼
FastAPI Service (feature engineering → standardization → inference)
      │
      ▼
car_price_model.pkl (loaded via joblib)
      │
      ▼
JSON Response: { predicted_price: 18500.0 }
```

### 🔮 Future Improvements

| Priority | Direction | Approach |
|----------|-----------|----------|
| 🔴 High | Model upgrade | XGBoost / LightGBM (expected R² ≥ 0.95) |
| 🔴 High | Feature engineering | Add vehicle age, mileage, accident history |
| 🟡 Medium | Data expansion | Scrape real-time used-car market prices |
| 🟡 Medium | Monitoring | Evidently AI / MLflow drift detection |
| 🟢 Low | Auto-retraining | Quarterly Airflow pipeline trigger |

In [ ]:
# ─────────────────────────────────────────────────────────────
#  6.1  Save Model Artifacts
# ─────────────────────────────────────────────────────────────
model_artifacts = {
    'sk_model':       sk_model,        # sklearn LinearRegression
    'ols_model':      ols_model,       # statsmodels OLS (contains p-values)
    'scaler':         scaler,          # fitted StandardScaler
    'final_features': final_features,  # list of selected feature names
    'feature_cols':   feature_cols,    # all feature names after OHE
    'test_metrics':   test_metrics,    # test set evaluation metrics
    'final_count':    final_count,     # number of final features
}

joblib.dump(model_artifacts, 'car_price_model.pkl')
print("✅ Model saved: car_price_model.pkl")

# Verify load
loaded = joblib.load('car_price_model.pkl')
assert loaded['final_count'] == final_count
print("✅ Load verification passed")

print(f"""
{'='*60}
  🎉 CRISP-DM Full Pipeline Complete!
{'='*60}

  Final Model Performance:
    R² (Test)    = {test_metrics['R2']:.4f}
    RMSE (Test)  = {test_metrics['RMSE']:,.2f} USD
    Features     = {final_count} (within 10–20 constraint)

  Output Files:
    ├─ eda_price_distribution.png
    ├─ eda_correlation.png
    ├─ eda_categorical_boxplots.png
    ├─ eda_scatter_plots.png
    ├─ eda_brand_price.png
    ├─ feature_selection_vif.png
    ├─ model_coefficients.png
    ├─ eval_actual_vs_predicted.png
    ├─ eval_residual_diagnostics.png
    ├─ eval_metrics_comparison.png
    └─ car_price_model.pkl
{'='*60}
""")